# 🤖 MODELADO PREDICTIVO

## Objetivo: Predecir si un jugador ganará el partido (`gano_partido`)

Compararemos tres modelos:
1. **Regresión Logística** (baseline simple)
2. **XGBoost** (gradient boosting)
3. **LightGBM** (gradient boosting optimizado)

## Preparación y selección de features

Preparamos los datos para modelado: seleccionamos features, codificamos variables categóricas y separamos X e y

In [229]:
from pathlib import Path

DATA_DIR = (Path('data') if Path('data').is_dir() else Path('../data')).resolve()

import joblib

MODEL_DIR = DATA_DIR.parent / 'models'

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                             f1_score, roc_auc_score, confusion_matrix, 
                             classification_report, roc_curve)
import xgboost as xgb
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

df_eda = pd.read_csv(DATA_DIR / 'processed' / 'Jugador_Partido_Features_Completo.csv')
df_equipos = pd.read_csv(DATA_DIR / 'processed' / 'Equipo_Partido_Features.csv')

# Preparar datos
print("="*80)
print("PREPARACIÓN DE DATOS")
print("="*80)

# Variable objetivo
y = df_eda['gano_partido']
print(f"\nDistribución de la variable objetivo:")
print(y.value_counts())
print(f"Balance: {y.mean():.1%} ganadores")

# Seleccionar features
# Excluir variables de identificación, resultados y la variable objetivo
features_excluir = ['player_name_normalized', 'partido_num', 'equipo', 'team_num', 
                    'puntos_team_1', 'puntos_team_2', 'ganador', 'gano_partido']

# Variables numéricas
features_numericas = [col for col in df_eda.columns 
                      if df_eda[col].dtype in ['int64', 'float64'] 
                      and col not in features_excluir]

# Variables categóricas
features_categoricas = [col for col in df_eda.columns 
                       if df_eda[col].dtype == 'object' 
                       and col not in features_excluir]

print(f"\n{len(features_numericas)} features numéricas:")
print(features_numericas)
print(f"\n{len(features_categoricas)} features categóricas:")
print(features_categoricas)

# Preparar X con encoding de categóricas
X = df_eda[features_numericas].copy()

# Renombrar features problemáticas (XGBoost no acepta <, >, [, ])
X = X.rename(columns={'player_speed_mps_<lambda_0>': 'player_speed_mps_quantile'})

# Label encoding para categóricas
label_encoders = {}
for col in features_categoricas:
    le = LabelEncoder()
    X[col] = le.fit_transform(df_eda[col].astype(str))
    label_encoders[col] = le

print(f"\nDimensiones finales: {X.shape}")
print(f"Total features: {X.shape[1]}")

PREPARACIÓN DE DATOS

Distribución de la variable objetivo:
gano_partido
1    54
0    53
Name: count, dtype: int64
Balance: 50.5% ganadores

25 features numéricas:
['velocidad_media_mps', 'player_speed_mps_<lambda_0>', 'velocidad_std_mps', 'distancia_total_m', 'distancia_media_por_frame_m', 'desplazamiento_std_m', 'distancia_media_pelota_m', 'distancia_min_pelota_m', 'distancia_media_red_m', 'distancia_media_companero_m', 'num_golpes', 'tiempo_promedio_desde_ultimo_golpe_s', 'aceleracion_media_mps2', 'aceleracion_std_mps2', 'duracion_total_s', 'frames_cerca_pelota', 'frames_zona_ofensiva', 'frames_zona_defensiva', 'total_frames', 'pct_frames_cerca_pelota', 'pct_tiempo_zona_ofensiva', 'tiempo_zona_defensiva_s', 'EDAD', 'ESTATURA', 'TALLA']

8 features categóricas:
['GENERO', 'EXPERIENCIA_PADEL', 'TIEMPO_JUGANDO_PADEL', 'NIVEL_ACTUAL_PADEL', 'ESTADO_FISICO', 'FRECUENCIA_DEPORTE', 'MANO_DOMINANTE', 'PRACTICA_OTRO_DEPORTE_RAQUETA']

Dimensiones finales: (107, 33)
Total features: 33


## Split train/test y escalado de features

Dividimos los datos en conjuntos de entrenamiento y prueba, y escalamos las features numéricas

In [230]:
# Split train/test estratificado
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print("\n" + "="*80)
print("SPLIT DE DATOS")
print("="*80)
print(f"\nTrain set: {len(X_train)} registros ({len(X_train)/len(X)*100:.1f}%)")
print(f"  - Ganadores: {y_train.sum()} ({y_train.mean():.1%})")
print(f"  - Perdedores: {(~y_train.astype(bool)).sum()} ({1-y_train.mean():.1%})")

print(f"\nTest set: {len(X_test)} registros ({len(X_test)/len(X)*100:.1f}%)")
print(f"  - Ganadores: {y_test.sum()} ({y_test.mean():.1%})")
print(f"  - Perdedores: {(~y_test.astype(bool)).sum()} ({1-y_test.mean():.1%})")

# Escalar features (importante para Logistic Regression)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"\n✅ Features escaladas para modelos lineales")


SPLIT DE DATOS

Train set: 80 registros (74.8%)
  - Ganadores: 40 (50.0%)
  - Perdedores: 40 (50.0%)

Test set: 27 registros (25.2%)
  - Ganadores: 14 (51.9%)
  - Perdedores: 13 (48.1%)

✅ Features escaladas para modelos lineales


### 2. Modelo Baseline: Regresión Logística

## Entrenamiento y evaluación: Regresión Logística

Entrenamos el modelo baseline de regresión logística y evaluamos su desempeño

In [231]:
# Entrenar Regresión Logística
lr_model = LogisticRegression(random_state=42, max_iter=1000, class_weight='balanced')
lr_model.fit(X_train_scaled, y_train)

# Predicciones
y_pred_lr = lr_model.predict(X_test_scaled)
y_pred_proba_lr = lr_model.predict_proba(X_test_scaled)[:, 1]

# Cross-validation
cv_scores_lr = cross_val_score(lr_model, X_train_scaled, y_train, 
                                cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
                                scoring='roc_auc')

print("="*80)
print("REGRESIÓN LOGÍSTICA (Baseline)")
print("="*80)
print(f"\n📊 Cross-validation ROC-AUC: {cv_scores_lr.mean():.4f} (+/- {cv_scores_lr.std():.4f})")
print(f"\n🎯 Métricas en Test Set:")
print(f"  Accuracy:  {accuracy_score(y_test, y_pred_lr):.4f}")
print(f"  Precision: {precision_score(y_test, y_pred_lr):.4f}")
print(f"  Recall:    {recall_score(y_test, y_pred_lr):.4f}")
print(f"  F1-Score:  {f1_score(y_test, y_pred_lr):.4f}")
print(f"  ROC-AUC:   {roc_auc_score(y_test, y_pred_proba_lr):.4f}")

print(f"\n📈 Confusion Matrix:")
cm_lr = confusion_matrix(y_test, y_pred_lr)
print(cm_lr)

# Features más importantes
feature_importance_lr = pd.DataFrame({
    'feature': X.columns,
    'coefficient': np.abs(lr_model.coef_[0])
}).sort_values('coefficient', ascending=False)

print(f"\n🔝 Top 10 Features (Regresión Logística):")
print(feature_importance_lr.head(10).to_string(index=False))

REGRESIÓN LOGÍSTICA (Baseline)

📊 Cross-validation ROC-AUC: 0.6656 (+/- 0.1276)

🎯 Métricas en Test Set:
  Accuracy:  0.7037
  Precision: 0.6875
  Recall:    0.7857
  F1-Score:  0.7333
  ROC-AUC:   0.7143

📈 Confusion Matrix:
[[ 8  5]
 [ 3 11]]

🔝 Top 10 Features (Regresión Logística):
                             feature  coefficient
            distancia_media_pelota_m     1.540360
                  NIVEL_ACTUAL_PADEL     1.464937
                   distancia_total_m     1.123058
                   EXPERIENCIA_PADEL     1.030476
                 velocidad_media_mps     0.846075
                               TALLA     0.644272
                      MANO_DOMINANTE     0.571927
tiempo_promedio_desde_ultimo_golpe_s     0.557947
                                EDAD     0.417776
                desplazamiento_std_m     0.346093


## Entrenamiento y evaluación: XGBoost

Entrenamos el modelo XGBoost con gradient boosting y evaluamos su desempeño

In [232]:
# Entrenar XGBoost
xgb_model = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=4,
    learning_rate=0.1,
    random_state=42,
    eval_metric='logloss',
    scale_pos_weight=len(y_train[y_train==0])/len(y_train[y_train==1])  # balance de clases
)
xgb_model.fit(X_train, y_train)

# Predicciones
y_pred_xgb = xgb_model.predict(X_test)
y_pred_proba_xgb = xgb_model.predict_proba(X_test)[:, 1]

# Cross-validation
cv_scores_xgb = cross_val_score(xgb_model, X_train, y_train,
                                 cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
                                 scoring='roc_auc')

print("="*80)
print("XGBOOST")
print("="*80)
print(f"\n📊 Cross-validation ROC-AUC: {cv_scores_xgb.mean():.4f} (+/- {cv_scores_xgb.std():.4f})")
print(f"\n🎯 Métricas en Test Set:")
print(f"  Accuracy:  {accuracy_score(y_test, y_pred_xgb):.4f}")
print(f"  Precision: {precision_score(y_test, y_pred_xgb):.4f}")
print(f"  Recall:    {recall_score(y_test, y_pred_xgb):.4f}")
print(f"  F1-Score:  {f1_score(y_test, y_pred_xgb):.4f}")
print(f"  ROC-AUC:   {roc_auc_score(y_test, y_pred_proba_xgb):.4f}")

print(f"\n📈 Confusion Matrix:")
cm_xgb = confusion_matrix(y_test, y_pred_xgb)
print(cm_xgb)

# Feature importance
feature_importance_xgb = pd.DataFrame({
    'feature': X.columns,
    'importance': xgb_model.feature_importances_
}).sort_values('importance', ascending=False)

print(f"\n🔝 Top 10 Features (XGBoost):")
print(feature_importance_xgb.head(10).to_string(index=False))

XGBOOST

📊 Cross-validation ROC-AUC: 0.6375 (+/- 0.1858)

🎯 Métricas en Test Set:
  Accuracy:  0.4815
  Precision: 0.5000
  Recall:    0.6429
  F1-Score:  0.5625
  ROC-AUC:   0.5934

📈 Confusion Matrix:
[[4 9]
 [5 9]]

🔝 Top 10 Features (XGBoost):
                 feature  importance
        duracion_total_s    0.096110
                   TALLA    0.091686
distancia_media_pelota_m    0.085979
                ESTATURA    0.075887
                    EDAD    0.068952
      NIVEL_ACTUAL_PADEL    0.065331
       distancia_total_m    0.052355
  aceleracion_media_mps2    0.043334
  distancia_min_pelota_m    0.041436
       velocidad_std_mps    0.041334


## Entrenamiento y evaluación: LightGBM

Entrenamos el modelo LightGBM con gradient boosting optimizado y evaluamos su desempeño

In [233]:
# Entrenar LightGBM
lgb_model = lgb.LGBMClassifier(
    n_estimators=100,
    max_depth=4,
    learning_rate=0.1,
    random_state=42,
    class_weight='balanced',
    verbose=-1
)
lgb_model.fit(X_train, y_train)

# Predicciones
y_pred_lgb = lgb_model.predict(X_test)
y_pred_proba_lgb = lgb_model.predict_proba(X_test)[:, 1]

# Cross-validation
cv_scores_lgb = cross_val_score(lgb_model, X_train, y_train,
                                 cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
                                 scoring='roc_auc')

print("="*80)
print("LIGHTGBM")
print("="*80)
print(f"\n📊 Cross-validation ROC-AUC: {cv_scores_lgb.mean():.4f} (+/- {cv_scores_lgb.std():.4f})")
print(f"\n🎯 Métricas en Test Set:")
print(f"  Accuracy:  {accuracy_score(y_test, y_pred_lgb):.4f}")
print(f"  Precision: {precision_score(y_test, y_pred_lgb):.4f}")
print(f"  Recall:    {recall_score(y_test, y_pred_lgb):.4f}")
print(f"  F1-Score:  {f1_score(y_test, y_pred_lgb):.4f}")
print(f"  ROC-AUC:   {roc_auc_score(y_test, y_pred_proba_lgb):.4f}")

print(f"\n📈 Confusion Matrix:")
cm_lgb = confusion_matrix(y_test, y_pred_lgb)
print(cm_lgb)

# Feature importance
feature_importance_lgb = pd.DataFrame({
    'feature': X.columns,
    'importance': lgb_model.feature_importances_
}).sort_values('importance', ascending=False)

print(f"\n🔝 Top 10 Features (LightGBM):")
print(feature_importance_lgb.head(10).to_string(index=False))

LIGHTGBM

📊 Cross-validation ROC-AUC: 0.6500 (+/- 0.1764)

🎯 Métricas en Test Set:
  Accuracy:  0.5185
  Precision: 0.5294
  Recall:    0.6429
  F1-Score:  0.5806
  ROC-AUC:   0.5659

📈 Confusion Matrix:
[[5 8]
 [5 9]]

🔝 Top 10 Features (LightGBM):
                    feature  importance
   distancia_media_pelota_m          32
          distancia_total_m          30
        velocidad_media_mps          26
      distancia_media_red_m          18
    pct_frames_cerca_pelota          16
        frames_cerca_pelota          13
                      TALLA          13
distancia_media_por_frame_m          13
          velocidad_std_mps          10
     distancia_min_pelota_m           9


---
## 🔧 HYPERPARAMETER TUNING

Optimizaremos los hiperparámetros de XGBoost y LightGBM usando RandomizedSearchCV para mejorar el rendimiento.

### 8. Hyperparameter Tuning - XGBoost

In [238]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import uniform, randint

# Grid de hiperparámetros para XGBoost
xgb_param_grid = {
    'n_estimators': randint(50, 300),
    'max_depth': randint(2, 8),
    'learning_rate': uniform(0.01, 0.3),
    'subsample': uniform(0.6, 0.4),
    'colsample_bytree': uniform(0.6, 0.4),
    'min_child_weight': randint(1, 10),
    'gamma': uniform(0, 0.5),
    'reg_alpha': uniform(0, 1),
    'reg_lambda': uniform(0, 1)
}

print("="*80)
print("HYPERPARAMETER TUNING - XGBOOST")
print("="*80)
print(f"\n🔍 Buscando mejores hiperparámetros...")
print(f"   Iteraciones: 50")
print(f"   CV folds: 5")
print(f"   Métrica: ROC-AUC")

# RandomizedSearchCV
xgb_random = RandomizedSearchCV(
    estimator=xgb.XGBClassifier(random_state=42, eval_metric='logloss'),
    param_distributions=xgb_param_grid,
    n_iter=50,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring='roc_auc',
    n_jobs=-1,
    random_state=42,
    verbose=1
)

xgb_random.fit(X_train, y_train)

print(f"\n✅ Búsqueda completada!")
print(f"\n🏆 Mejores hiperparámetros:")
for param, value in xgb_random.best_params_.items():
    print(f"   {param}: {value}")

print(f"\n📊 Mejor CV ROC-AUC: {xgb_random.best_score_:.4f}")

# Entrenar modelo optimizado
xgb_tuned = xgb_random.best_estimator_
y_pred_xgb_tuned = xgb_tuned.predict(X_test)
y_pred_proba_xgb_tuned = xgb_tuned.predict_proba(X_test)[:, 1]

print(f"\n🎯 Métricas en Test Set (XGBoost Optimizado):")
print(f"  Accuracy:  {accuracy_score(y_test, y_pred_xgb_tuned):.4f}")
print(f"  Precision: {precision_score(y_test, y_pred_xgb_tuned):.4f}")
print(f"  Recall:    {recall_score(y_test, y_pred_xgb_tuned):.4f}")
print(f"  F1-Score:  {f1_score(y_test, y_pred_xgb_tuned):.4f}")
print(f"  ROC-AUC:   {roc_auc_score(y_test, y_pred_proba_xgb_tuned):.4f}")

print(f"\n📈 Mejora vs XGBoost default:")
print(f"  ROC-AUC: {roc_auc_score(y_test, y_pred_proba_xgb_tuned) - roc_auc_score(y_test, y_pred_proba_xgb):.4f} ({((roc_auc_score(y_test, y_pred_proba_xgb_tuned) / roc_auc_score(y_test, y_pred_proba_xgb)) - 1) * 100:.1f}%)")

cm_xgb_tuned = confusion_matrix(y_test, y_pred_xgb_tuned)
print(f"\n📊 Confusion Matrix:")
print(cm_xgb_tuned)

HYPERPARAMETER TUNING - XGBOOST

🔍 Buscando mejores hiperparámetros...
   Iteraciones: 50
   CV folds: 5
   Métrica: ROC-AUC
Fitting 5 folds for each of 50 candidates, totalling 250 fits

✅ Búsqueda completada!

🏆 Mejores hiperparámetros:
   colsample_bytree: 0.8066785029706267
   gamma: 0.3285556642500834
   learning_rate: 0.1407018696033673
   max_depth: 6
   min_child_weight: 7
   n_estimators: 244
   reg_alpha: 0.24773098950115746
   reg_lambda: 0.3559726786512616
   subsample: 0.9031384441857476

📊 Mejor CV ROC-AUC: 0.6516

🎯 Métricas en Test Set (XGBoost Optimizado):
  Accuracy:  0.5185
  Precision: 0.5294
  Recall:    0.6429
  F1-Score:  0.5806
  ROC-AUC:   0.5440

📈 Mejora vs XGBoost default:
  ROC-AUC: -0.0495 (-8.3%)

📊 Confusion Matrix:
[[5 8]
 [5 9]]


## Optimización de hiperparámetros: XGBoost

Realizamos búsqueda aleatoria de hiperparámetros para optimizar XGBoost

### 9. Hyperparameter Tuning - LightGBM

In [239]:
# Grid de hiperparámetros para LightGBM
lgb_param_grid = {
    'n_estimators': randint(50, 300),
    'max_depth': randint(2, 8),
    'learning_rate': uniform(0.01, 0.3),
    'subsample': uniform(0.6, 0.4),
    'colsample_bytree': uniform(0.6, 0.4),
    'min_child_samples': randint(5, 50),
    'reg_alpha': uniform(0, 1),
    'reg_lambda': uniform(0, 1),
    'num_leaves': randint(10, 100)
}

print("="*80)
print("HYPERPARAMETER TUNING - LIGHTGBM")
print("="*80)
print(f"\n🔍 Buscando mejores hiperparámetros...")
print(f"   Iteraciones: 50")
print(f"   CV folds: 5")
print(f"   Métrica: ROC-AUC")

# RandomizedSearchCV
lgb_random = RandomizedSearchCV(
    estimator=lgb.LGBMClassifier(random_state=42, verbose=-1),
    param_distributions=lgb_param_grid,
    n_iter=50,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring='roc_auc',
    n_jobs=-1,
    random_state=42,
    verbose=1
)

lgb_random.fit(X_train, y_train)

print(f"\n✅ Búsqueda completada!")
print(f"\n🏆 Mejores hiperparámetros:")
for param, value in lgb_random.best_params_.items():
    print(f"   {param}: {value}")

print(f"\n📊 Mejor CV ROC-AUC: {lgb_random.best_score_:.4f}")

# Entrenar modelo optimizado
lgb_tuned = lgb_random.best_estimator_
y_pred_lgb_tuned = lgb_tuned.predict(X_test)
y_pred_proba_lgb_tuned = lgb_tuned.predict_proba(X_test)[:, 1]

print(f"\n🎯 Métricas en Test Set (LightGBM Optimizado):")
print(f"  Accuracy:  {accuracy_score(y_test, y_pred_lgb_tuned):.4f}")
print(f"  Precision: {precision_score(y_test, y_pred_lgb_tuned):.4f}")
print(f"  Recall:    {recall_score(y_test, y_pred_lgb_tuned):.4f}")
print(f"  F1-Score:  {f1_score(y_test, y_pred_lgb_tuned):.4f}")
print(f"  ROC-AUC:   {roc_auc_score(y_test, y_pred_proba_lgb_tuned):.4f}")

print(f"\n📈 Mejora vs LightGBM default:")
print(f"  ROC-AUC: {roc_auc_score(y_test, y_pred_proba_lgb_tuned) - roc_auc_score(y_test, y_pred_proba_lgb):.4f} ({((roc_auc_score(y_test, y_pred_proba_lgb_tuned) / roc_auc_score(y_test, y_pred_proba_lgb)) - 1) * 100:.1f}%)")

cm_lgb_tuned = confusion_matrix(y_test, y_pred_lgb_tuned)
print(f"\n📊 Confusion Matrix:")
print(cm_lgb_tuned)

HYPERPARAMETER TUNING - LIGHTGBM

🔍 Buscando mejores hiperparámetros...
   Iteraciones: 50
   CV folds: 5
   Métrica: ROC-AUC
Fitting 5 folds for each of 50 candidates, totalling 250 fits

✅ Búsqueda completada!

🏆 Mejores hiperparámetros:
   colsample_bytree: 0.996884623716487
   learning_rate: 0.19524445288831496
   max_depth: 3
   min_child_samples: 26
   n_estimators: 285
   num_leaves: 98
   reg_alpha: 0.2912291401980419
   reg_lambda: 0.6118528947223795
   subsample: 0.6557975442608167

📊 Mejor CV ROC-AUC: 0.6781

🎯 Métricas en Test Set (LightGBM Optimizado):
  Accuracy:  0.5556
  Precision: 0.5625
  Recall:    0.6429
  F1-Score:  0.6000
  ROC-AUC:   0.5385

📈 Mejora vs LightGBM default:
  ROC-AUC: -0.0275 (-4.9%)

📊 Confusion Matrix:
[[6 7]
 [5 9]]


## Optimización de hiperparámetros: LightGBM

Realizamos búsqueda aleatoria de hiperparámetros para optimizar LightGBM


# 🤖 MODELOS PREDICTIVOS - DATASET DE EQUIPOS

Entrenamiento de modelos para predecir victoria de equipos

In [ ]:
# Imports para modelos de equipos
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, roc_curve

print("✅ Librerías importadas para modelado de equipos")

✅ Librerías importadas para modelado de equipos


In [ ]:

# Preparacion de datos para modelado de equipos
print("="*80)
print("PREPARACION DE DATOS - MODELOS DE EQUIPOS")
print("="*80)

# Seleccionar features consistentes entre entrenamiento y simulacion
features_base_equipo = [
    'velocidad_media_equipo_mps',
    'distancia_total_equipo_m',
    'distancia_media_por_frame_m',
    'distancia_media_pelota_equipo_m',
    'golpes_totales_equipo',
    'aceleracion_media_mps2',
    'pct_frames_cerca_pelota_equipo',
    'pct_tiempo_zona_ofensiva',
    'frames_cerca_pelota',
    'frames_zona_ofensiva',
    'frames_zona_defensiva',
    'edad_promedio_equipo',
    'estatura_promedio_equipo',
    'talla_promedio_equipo'
]

features_diff_equipo = [
    'diff_golpes_vs_rival',
    'diff_distancia_pelota_vs_rival',
    'diff_pct_cerca_vs_rival',
    'diff_velocidad_vs_rival'
]

features_equipos = [f for f in features_base_equipo + features_diff_equipo if f in df_equipos.columns]
missing_features = list(set(features_base_equipo + features_diff_equipo) - set(features_equipos))
print(f"Features seleccionadas: {len(features_equipos)}")
print(f"   {features_equipos}")

if missing_features:
    print(f"Advertencia: faltan en df_equipos {missing_features}")

# Separar features y target
X_equipos = df_equipos[features_equipos].copy()
y_equipos = df_equipos['gano_partido'].copy()

# Guardar medias de entrenamiento para reutilizar en simulacion
feature_promedios_eq = X_equipos.mean()

# Manejar valores nulos en features diferenciales (imputar con 0)
print(f"? Valores nulos antes de imputar:")
print(X_equipos.isnull().sum()[X_equipos.isnull().sum() > 0])

X_equipos = X_equipos.fillna(0)

print(f"? Valores nulos despues de imputar: {X_equipos.isnull().sum().sum()}")      

# Train/test split estratificado
X_train_eq, X_test_eq, y_train_eq, y_test_eq = train_test_split(
    X_equipos, y_equipos, test_size=0.25, random_state=42, stratify=y_equipos
)

print(f"? Division del dataset:")
print(f"   Train: {len(X_train_eq)} registros ({len(X_train_eq)/len(X_equipos):.1%})")
print(f"   Test:  {len(X_test_eq)} registros ({len(X_test_eq)/len(X_equipos):.1%})")
print(f"   Distribucion Train - Ganadores: {y_train_eq.sum()} ({y_train_eq.mean():.1%})")
print(f"   Distribucion Test  - Ganadores: {y_test_eq.sum()} ({y_test_eq.mean():.1%})")

# Escalar features
scaler_eq = StandardScaler()
X_train_eq_scaled = scaler_eq.fit_transform(X_train_eq)
X_test_eq_scaled = scaler_eq.transform(X_test_eq)

print(f"? Datos escalados y listos para entrenamiento")

PREPARACION DE DATOS - MODELOS DE EQUIPOS
Features seleccionadas: 18
   ['velocidad_media_equipo_mps', 'distancia_total_equipo_m', 'distancia_media_por_frame_m', 'distancia_media_pelota_equipo_m', 'golpes_totales_equipo', 'aceleracion_media_mps2', 'pct_frames_cerca_pelota_equipo', 'pct_tiempo_zona_ofensiva', 'frames_cerca_pelota', 'frames_zona_ofensiva', 'frames_zona_defensiva', 'edad_promedio_equipo', 'estatura_promedio_equipo', 'talla_promedio_equipo', 'diff_golpes_vs_rival', 'diff_distancia_pelota_vs_rival', 'diff_pct_cerca_vs_rival', 'diff_velocidad_vs_rival']
? Valores nulos antes de imputar:
diff_golpes_vs_rival              10
diff_distancia_pelota_vs_rival    10
diff_pct_cerca_vs_rival           10
diff_velocidad_vs_rival           10
dtype: int64
? Valores nulos despues de imputar: 0
? Division del dataset:
   Train: 40 registros (74.1%)
   Test:  14 registros (25.9%)
   Distribucion Train - Ganadores: 20 (50.0%)
   Distribucion Test  - Ganadores: 7 (50.0%)
? Datos escalados y

In [ ]:
# 1. Regresión Logística
print("="*80)
print("MODELO 1: REGRESIÓN LOGÍSTICA (BASELINE)")
print("="*80)

lr_model_eq = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
lr_model_eq.fit(X_train_eq_scaled, y_train_eq)

# Predicciones
y_pred_lr_eq = lr_model_eq.predict(X_test_eq_scaled)
y_pred_proba_lr_eq = lr_model_eq.predict_proba(X_test_eq_scaled)[:, 1]

# Métricas
acc_lr_eq = accuracy_score(y_test_eq, y_pred_lr_eq)
precision_lr_eq = precision_score(y_test_eq, y_pred_lr_eq)
recall_lr_eq = recall_score(y_test_eq, y_pred_lr_eq)
f1_lr_eq = f1_score(y_test_eq, y_pred_lr_eq)
roc_auc_lr_eq = roc_auc_score(y_test_eq, y_pred_proba_lr_eq)

print(f"\n📊 Métricas en Test Set:")
print(f"   Accuracy:  {acc_lr_eq:.3f}")
print(f"   Precision: {precision_lr_eq:.3f}")
print(f"   Recall:    {recall_lr_eq:.3f}")
print(f"   F1-Score:  {f1_lr_eq:.3f}")
print(f"   ROC-AUC:   {roc_auc_lr_eq:.3f}")

# Cross-validation
cv_scores_lr_eq = cross_val_score(lr_model_eq, X_train_eq_scaled, y_train_eq, 
                                   cv=5, scoring='roc_auc')
print(f"\n📊 Cross-Validation (5-fold):")
print(f"   ROC-AUC medio: {cv_scores_lr_eq.mean():.3f} (+/- {cv_scores_lr_eq.std():.3f})")

# Top features
feature_importance_lr_eq = pd.DataFrame({
    'feature': features_equipos,
    'coefficient': np.abs(lr_model_eq.coef_[0])
}).sort_values('coefficient', ascending=False)

print(f"\n🔝 Top 10 Features más importantes:")
print(feature_importance_lr_eq.head(10).to_string(index=False))

MODELO 1: REGRESIÓN LOGÍSTICA (BASELINE)

📊 Métricas en Test Set:
   Accuracy:  0.429
   Precision: 0.444
   Recall:    0.571
   F1-Score:  0.500
   ROC-AUC:   0.612

📊 Cross-Validation (5-fold):
   ROC-AUC medio: 0.637 (+/- 0.083)

🔝 Top 10 Features más importantes:
                        feature  coefficient
 diff_distancia_pelota_vs_rival     1.092608
           edad_promedio_equipo     0.929198
        diff_velocidad_vs_rival     0.674251
distancia_media_pelota_equipo_m     0.652025
       estatura_promedio_equipo     0.647214
         aceleracion_media_mps2     0.552736
          talla_promedio_equipo     0.455668
           diff_golpes_vs_rival     0.356552
        diff_pct_cerca_vs_rival     0.349492
    distancia_media_por_frame_m     0.333509


In [ ]:
# 2. XGBoost
print("="*80)
print("MODELO 2: XGBOOST")
print("="*80)

xgb_model_eq = XGBClassifier(
    n_estimators=100,
    max_depth=4,
    learning_rate=0.1,
    random_state=42,
    eval_metric='logloss'
)
xgb_model_eq.fit(X_train_eq, y_train_eq)

# Predicciones
y_pred_xgb_eq = xgb_model_eq.predict(X_test_eq)
y_pred_proba_xgb_eq = xgb_model_eq.predict_proba(X_test_eq)[:, 1]

# Métricas
acc_xgb_eq = accuracy_score(y_test_eq, y_pred_xgb_eq)
precision_xgb_eq = precision_score(y_test_eq, y_pred_xgb_eq)
recall_xgb_eq = recall_score(y_test_eq, y_pred_xgb_eq)
f1_xgb_eq = f1_score(y_test_eq, y_pred_xgb_eq)
roc_auc_xgb_eq = roc_auc_score(y_test_eq, y_pred_proba_xgb_eq)

print(f"\n📊 Métricas en Test Set:")
print(f"   Accuracy:  {acc_xgb_eq:.3f}")
print(f"   Precision: {precision_xgb_eq:.3f}")
print(f"   Recall:    {recall_xgb_eq:.3f}")
print(f"   F1-Score:  {f1_xgb_eq:.3f}")
print(f"   ROC-AUC:   {roc_auc_xgb_eq:.3f}")

# Cross-validation
cv_scores_xgb_eq = cross_val_score(xgb_model_eq, X_train_eq, y_train_eq, 
                                    cv=5, scoring='roc_auc')
print(f"\n📊 Cross-Validation (5-fold):")
print(f"   ROC-AUC medio: {cv_scores_xgb_eq.mean():.3f} (+/- {cv_scores_xgb_eq.std():.3f})")

# Feature importance
feature_importance_xgb_eq = pd.DataFrame({
    'feature': features_equipos,
    'importance': xgb_model_eq.feature_importances_
}).sort_values('importance', ascending=False)

print(f"\n🔝 Top 10 Features más importantes:")
print(feature_importance_xgb_eq.head(10).to_string(index=False))

MODELO 2: XGBOOST

📊 Métricas en Test Set:
   Accuracy:  0.571
   Precision: 0.571
   Recall:    0.571
   F1-Score:  0.571
   ROC-AUC:   0.612

📊 Cross-Validation (5-fold):
   ROC-AUC medio: 0.463 (+/- 0.219)

🔝 Top 10 Features más importantes:
                        feature  importance
 pct_frames_cerca_pelota_equipo    0.140710
           edad_promedio_equipo    0.132887
          talla_promedio_equipo    0.129657
distancia_media_pelota_equipo_m    0.093016
 diff_distancia_pelota_vs_rival    0.092226
         aceleracion_media_mps2    0.077629
          golpes_totales_equipo    0.057506
        diff_pct_cerca_vs_rival    0.055734
       distancia_total_equipo_m    0.042707
            frames_cerca_pelota    0.036797


In [ ]:
# 3. LightGBM
print("="*80)
print("MODELO 3: LIGHTGBM")
print("="*80)

lgb_model_eq = LGBMClassifier(
    n_estimators=100,
    max_depth=4,
    learning_rate=0.1,
    random_state=42,
    verbose=-1
)
lgb_model_eq.fit(X_train_eq, y_train_eq)

# Predicciones
y_pred_lgb_eq = lgb_model_eq.predict(X_test_eq)
y_pred_proba_lgb_eq = lgb_model_eq.predict_proba(X_test_eq)[:, 1]

# Métricas
acc_lgb_eq = accuracy_score(y_test_eq, y_pred_lgb_eq)
precision_lgb_eq = precision_score(y_test_eq, y_pred_lgb_eq)
recall_lgb_eq = recall_score(y_test_eq, y_pred_lgb_eq)
f1_lgb_eq = f1_score(y_test_eq, y_pred_lgb_eq)
roc_auc_lgb_eq = roc_auc_score(y_test_eq, y_pred_proba_lgb_eq)

print(f"\n📊 Métricas en Test Set:")
print(f"   Accuracy:  {acc_lgb_eq:.3f}")
print(f"   Precision: {precision_lgb_eq:.3f}")
print(f"   Recall:    {recall_lgb_eq:.3f}")
print(f"   F1-Score:  {f1_lgb_eq:.3f}")
print(f"   ROC-AUC:   {roc_auc_lgb_eq:.3f}")

# Cross-validation
cv_scores_lgb_eq = cross_val_score(lgb_model_eq, X_train_eq, y_train_eq, 
                                    cv=5, scoring='roc_auc')
print(f"\n📊 Cross-Validation (5-fold):")
print(f"   ROC-AUC medio: {cv_scores_lgb_eq.mean():.3f} (+/- {cv_scores_lgb_eq.std():.3f})")

# Feature importance
feature_importance_lgb_eq = pd.DataFrame({
    'feature': features_equipos,
    'importance': lgb_model_eq.feature_importances_
}).sort_values('importance', ascending=False)

print(f"\n🔝 Top 10 Features más importantes:")
print(feature_importance_lgb_eq.head(10).to_string(index=False))

MODELO 3: LIGHTGBM

📊 Métricas en Test Set:
   Accuracy:  0.571
   Precision: 0.556
   Recall:    0.714
   F1-Score:  0.625
   ROC-AUC:   0.571

📊 Cross-Validation (5-fold):
   ROC-AUC medio: 0.500 (+/- 0.000)

🔝 Top 10 Features más importantes:
                       feature  importance
         golpes_totales_equipo         100
    velocidad_media_equipo_mps           0
         frames_zona_defensiva           0
       diff_pct_cerca_vs_rival           0
diff_distancia_pelota_vs_rival           0
          diff_golpes_vs_rival           0
         talla_promedio_equipo           0
      estatura_promedio_equipo           0
          edad_promedio_equipo           0
          frames_zona_ofensiva           0


In [ ]:
MODEL_DIR.mkdir(parents=True, exist_ok=True)

joblib.dump({
    'X': X,
    'X_equipos': X_equipos,
    'X_test': X_test,
    'X_test_eq': X_test_eq,
    'X_train': X_train,
    'X_train_eq': X_train_eq,
    'acc_lgb_eq': acc_lgb_eq,
    'acc_lr_eq': acc_lr_eq,
    'acc_xgb_eq': acc_xgb_eq,
    'cm_lgb': cm_lgb,
    'cm_lr': cm_lr,
    'cm_xgb': cm_xgb,
    'cv_scores_lgb': cv_scores_lgb,
    'cv_scores_lgb_eq': cv_scores_lgb_eq,
    'cv_scores_lr': cv_scores_lr,
    'cv_scores_lr_eq': cv_scores_lr_eq,
    'cv_scores_xgb': cv_scores_xgb,
    'cv_scores_xgb_eq': cv_scores_xgb_eq,
    'f1_lgb_eq': f1_lgb_eq,
    'f1_lr_eq': f1_lr_eq,
    'f1_xgb_eq': f1_xgb_eq,
    'feature_importance_lgb': feature_importance_lgb,
    'feature_importance_lgb_eq': feature_importance_lgb_eq,
    'feature_importance_lr': feature_importance_lr,
    'feature_importance_lr_eq': feature_importance_lr_eq,
    'feature_importance_xgb': feature_importance_xgb,
    'feature_importance_xgb_eq': feature_importance_xgb_eq,
    'feature_promedios_eq': feature_promedios_eq,
    'features_categoricas': features_categoricas,
    'features_equipos': features_equipos,
    'features_numericas': features_numericas,
    'label_encoders': label_encoders,
    'lgb_random': lgb_random,
    'lr_model': lr_model,
    'lr_model_eq': lr_model_eq,
    'precision_lgb_eq': precision_lgb_eq,
    'precision_lr_eq': precision_lr_eq,
    'precision_xgb_eq': precision_xgb_eq,
    'recall_lgb_eq': recall_lgb_eq,
    'recall_lr_eq': recall_lr_eq,
    'recall_xgb_eq': recall_xgb_eq,
    'roc_auc_lgb_eq': roc_auc_lgb_eq,
    'roc_auc_lr_eq': roc_auc_lr_eq,
    'roc_auc_xgb_eq': roc_auc_xgb_eq,
    'scaler': scaler,
    'scaler_eq': scaler_eq,
    'xgb_random': xgb_random,
    'y': y,
    'y_pred_lgb': y_pred_lgb,
    'y_pred_lgb_eq': y_pred_lgb_eq,
    'y_pred_lgb_tuned': y_pred_lgb_tuned,
    'y_pred_lr': y_pred_lr,
    'y_pred_lr_eq': y_pred_lr_eq,
    'y_pred_proba_lgb': y_pred_proba_lgb,
    'y_pred_proba_lgb_eq': y_pred_proba_lgb_eq,
    'y_pred_proba_lgb_tuned': y_pred_proba_lgb_tuned,
    'y_pred_proba_lr': y_pred_proba_lr,
    'y_pred_proba_lr_eq': y_pred_proba_lr_eq,
    'y_pred_proba_xgb': y_pred_proba_xgb,
    'y_pred_proba_xgb_eq': y_pred_proba_xgb_eq,
    'y_pred_proba_xgb_tuned': y_pred_proba_xgb_tuned,
    'y_pred_xgb': y_pred_xgb,
    'y_pred_xgb_eq': y_pred_xgb_eq,
    'y_pred_xgb_tuned': y_pred_xgb_tuned,
    'y_test': y_test,
    'y_test_eq': y_test_eq,
}, MODEL_DIR / 'estado_modelos.joblib')
